# 02 — Data Preparation

Giai đoạn 3 của CRISP-DM: làm sạch theo phát hiện ở GIAI ĐOẠN 2, và xây
3 bảng đặc trưng phục vụ GIAI ĐOẠN 4 (luật kết hợp, phân cụm tỉnh, phát
hiện bất thường). Toàn bộ logic nằm trong `src/features.py`; xem
`reports/decisions_log.md` (DEC-005, DEC-006, DEC-007) cho các quyết
định và lý do.

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import yaml

from src.preprocessing import load_raw_csv, SUBJECT_COLUMNS
from src.features import (
    filter_standard_candidates, build_transactions_df,
    build_province_features, scale_features,
    build_site_features, build_sliding_window_features,
)

cfg = yaml.safe_load(open("../config.yaml", encoding="utf-8"))
edges = cfg["discretization"]["edges"]
labels = cfg["discretization"]["labels"]
print("Discretization edges:", edges, "labels:", labels)

df = load_raw_csv("../data/raw/diem_thi_2026.csv")
print("Tổng số thí sinh:", f"{len(df):,}")

Discretization edges: [0, 5, 6.5, 8, 10.01] labels: ['Yeu', 'TrungBinh', 'Kha', 'Gioi']


Tổng số thí sinh: 1,208,863


## 3.1 Lọc thí sinh "chuẩn" cho bảng giao dịch (DEC-005)

Chỉ giữ thí sinh thi đúng 4 môn (2 bắt buộc + 2 tự chọn) — loại nhóm có
hồ sơ thi khác thường (thiếu môn bắt buộc, thi <4 môn) để không làm
nhiễu luật kết hợp.

In [2]:
std_df = filter_standard_candidates(df)
n_excluded = len(df) - len(std_df)
print(f"Giữ lại: {len(std_df):,} ({100*len(std_df)/len(df):.2f}%)")
print(f"Loại bỏ: {n_excluded:,} ({100*n_excluded/len(df):.2f}%)")

Giữ lại: 1,165,582 (96.42%)
Loại bỏ: 43,281 (3.58%)


## 3.2 Bảng giao dịch cho luật kết hợp (4.1)

Mỗi thí sinh = 1 giao dịch. Item gồm 2 loại:
- `Chon_<Mon>`: đã chọn thi môn tự chọn đó.
- `<Mon>_<Muc>`: mức điểm rời rạc hóa theo quy ước Yếu/TB/Khá/Giỏi (DEC-006).

Lưu dạng one-hot (bool) để dùng trực tiếp với `mlxtend.frequent_patterns`.

In [3]:
tx_long = build_transactions_df(std_df, edges, labels)
onehot = pd.crosstab(tx_long["sbd"], tx_long["item"]).astype(bool)
print("Shape one-hot:", onehot.shape)
print("Số item (cột) duy nhất:", onehot.shape[1])
onehot.sum().sort_values(ascending=False).head(10)

Shape one-hot: (1165582, 58)
Số item (cột) duy nhất: 58


item
Chon_LichSu         546867
NguVan_Kha          534604
Toan_Yeu            447870
Chon_DiaLy          429681
Chon_VatLy          374758
NguVan_TrungBinh    344222
Chon_NgoaiNgu       337241
Toan_TrungBinh      298736
Chon_GDKTPL         278629
Chon_HoaHoc         241958
dtype: int64

In [4]:
onehot.to_parquet("../data/processed/transactions_onehot.parquet")
print("Đã lưu data/processed/transactions_onehot.parquet")

Đã lưu data/processed/transactions_onehot.parquet


## 3.3 Đặc trưng cấp tỉnh cho phân cụm (4.2)

34 dòng (1 tỉnh/dòng): mean/median/std/tỷ lệ giỏi/tỷ lệ yếu mỗi môn +
tỷ lệ chọn từng môn tự chọn. Dùng toàn bộ thí sinh (thống kê mô tả không
yêu cầu đủ 4 môn). Chuẩn hóa StandardScaler để chuẩn bị cho K-Means/
Hierarchical/DBSCAN (nhạy với thang đo).

In [5]:
province_feat = build_province_features(df)
print("Shape:", province_feat.shape)
province_feat[["n_candidates", "Toan_mean", "NguVan_mean", "VatLy_pct_chon", "LichSu_pct_chon"]].round(2)

Shape: (34, 71)


,n_candidates,Toan_mean,NguVan_mean,VatLy_pct_chon,LichSu_pct_chon
province_name,,,,,
An Giang,39861,5.29,6.35,0.30,0.51
Bắc Ninh,43812,5.82,6.34,0.32,0.48
Cao Bằng,5607,4.76,6.04,0.23,0.70
Cà Mau,19534,5.15,6.56,0.30,0.54
Cần Thơ,34071,5.07,6.36,0.30,0.48
Gia Lai,37378,5.43,6.17,0.34,0.49
Huế,14087,5.80,6.52,0.32,0.42
Hà Nội,126867,6.16,7.02,0.30,0.46
Hà Tĩnh,18487,5.98,6.98,0.25,0.52


In [6]:
scale_cols = [c for c in province_feat.columns if c != "n_candidates"]
province_scaled, scaler = scale_features(province_feat, columns=scale_cols)

province_feat.to_parquet("../data/processed/province_features.parquet")
province_scaled.to_parquet("../data/processed/province_features_scaled.parquet")
print("Đã lưu 2 file: province_features(_scaled).parquet")
print("Scaled mean ~0, std ~1:", province_scaled.mean().abs().max(), province_scaled.std().round(2).unique()[:3])

Đã lưu 2 file: province_features(_scaled).parquet
Scaled mean ~0, std ~1: 1.6424770040778052e-15 [1.02]


## 3.4 Đặc trưng cấp "điểm thi" cho phát hiện bất thường (4.3)

Đơn vị chính: nhóm theo 3 chữ số giữa của SBD (xem DEC-003, DEC-007—
thực chất là khối 1.000 SBD liên tiếp, không hẳn là 1 phòng thi vật lý).
Chỉ giữ nhóm ≥10 thí sinh. Đặc trưng: mean/std mỗi môn, chênh lệch so
với TB tỉnh, tỷ lệ trùng điểm ở mức phổ biến nhất, entropy phân bố
điểm.

In [7]:
site_feat = build_site_features(df, SUBJECT_COLUMNS, min_group_size=10)
print("Số nhóm (site_id):", len(site_feat))
print(site_feat["n_candidates"].describe())
site_feat.to_parquet("../data/processed/site_features.parquet")

Số nhóm (site_id): 1237
count    1237.00000
mean      977.25384
std       106.22932
min        11.00000
25%       992.00000
50%       996.00000
75%       998.00000
max      1000.00000
Name: n_candidates, dtype: float64


### Phân tích độ nhạy: cửa sổ trượt thay cho "mã điểm thi"

Theo yêu cầu, thử thêm phương án nhóm theo cửa sổ trượt (24/50/100 SBD
liên tiếp trong cùng tỉnh — gần quy mô phòng thi thật hơn) để đối chiếu
độ nhạy của kết quả phát hiện bất thường với cách định nghĩa nhóm.

In [8]:
for w in cfg["exam_site"]["sliding_window_sizes"]:
    win_feat = build_sliding_window_features(df, SUBJECT_COLUMNS, window_size=w)
    win_feat.to_parquet(f"../data/processed/site_features_window{w}.parquet")
    print(f"window={w}: {len(win_feat):,} nhóm, trung vị kích thước = {win_feat['n_candidates'].median():.0f}")

window=24: 50,369 nhóm, trung vị kích thước = 24


window=50: 24,179 nhóm, trung vị kích thước = 50


window=100: 12,091 nhóm, trung vị kích thước = 100


## 3.5 Tóm tắt dữ liệu đã lưu (`data/processed/`)

| File | Dùng cho | Shape |
|---|---|---|
| `transactions_onehot.parquet` | 4.1 Luật kết hợp (FP-Growth/Apriori) | thí sinh × item |
| `province_features.parquet` / `_scaled.parquet` | 4.2 Phân cụm tỉnh | 34 × đặc trưng |
| `site_features.parquet` | 4.3 Phát hiện bất thường (đơn vị chính) | ~1.237 × đặc trưng |
| `site_features_window{24,50,100}.parquet` | 4.3 Phân tích độ nhạy | nhiều kích thước |

Tất cả được tính từ dữ liệu thật trong notebook này — xem số liệu cụ thể
ở các cell phía trên.